# Experiment 9: One-Way and Two-Way ANOVA

**Date:** April 17, 2026  
**Experiment Title:** One-Way And Two-Way ANOVA  
**Software:** Jupyter Notebook  

## Aim
To implement one-way and two-way ANOVA to analyze plant growth under different conditions.

## Theory
**Tukey's HSD (Honest Significant Difference) Test:** A post-hoc test used after ANOVA to identify which specific treatment groups have statistically significant differences. It controls for Type I error when making multiple comparisons between group means. If p < 0.05, the difference between treatments is statistically significant.

In [2]:
# Import required libraries
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import warnings
warnings.filterwarnings('ignore')

# Import statsmodels components
try:
    import statsmodels.formula.api as smf
    from statsmodels.stats.anova import anova_lm
except ImportError:
    # Handle version compatibility issues
    from statsmodels.regression.linear_model import OLS
    from statsmodels.tools.tools import add_constant
    print("Note: Using alternative statsmodels import method")

Note: Using alternative statsmodels import method


## Part 1: One-Way ANOVA Analysis

**Objective:** Determine if treatments are effective on plant growth and identify which treatments differ significantly.

### Hypotheses:
- **Null Hypothesis (H₀):** There is no significant difference in plant growth across treatments.
- **Alternate Hypothesis (H₁):** At least one treatment leads to a significant difference in plant growth.

In [3]:
def analyze_plant_growth(df):
    """
    Performs One-Way ANOVA and Tukey's HSD test on plant growth data.
    
    Parameters:
    df (pd.DataFrame): DataFrame with 'Treatment' and 'Height' columns
    """
    print("=" * 80)
    print("ONE-WAY ANOVA ANALYSIS")
    print("=" * 80)
    
    # Step 1: Group data by Treatment
    grouped_data = df.groupby('Treatment')['Height'].apply(list)
    print("\n1. GROUPED DATA BY TREATMENT:")
    print("-" * 80)
    for treatment, heights in grouped_data.items():
        print(f"{treatment}: {heights}")
    
    # Step 2: Convert lists to NumPy arrays
    samples = [np.array(heights) for heights in grouped_data.values]
    
    # Step 3: Perform One-Way ANOVA
    print("\n2. PERFORMING ONE-WAY ANOVA:")
    print("-" * 80)
    anova_stat, anova_p = stats.f_oneway(*samples)
    
    print(f"F-Statistic: {anova_stat:.6f}")
    print(f"P-value: {anova_p:.6f}")
    
    # Interpretation
    print("\n3. INTERPRETATION OF ONE-WAY ANOVA RESULTS:")
    print("-" * 80)
    if anova_p < 0.05:
        print(f"✓ P-value ({anova_p:.6f}) < 0.05")
        print("✓ Reject the null hypothesis")
        print("✓ CONCLUSION: At least one treatment significantly affects plant growth.")
    else:
        print(f"✗ P-value ({anova_p:.6f}) >= 0.05")
        print("✗ Fail to reject the null hypothesis")
        print("✗ CONCLUSION: No significant difference in plant growth across treatments.")
    
    # Step 4: Perform Tukey's HSD Test
    print("\n4. TUKEY'S HSD (HONEST SIGNIFICANT DIFFERENCE) TEST:")
    print("-" * 80)
    tukey_result = pairwise_tukeyhsd(endog=df['Height'], groups=df['Treatment'], alpha=0.05)
    print(tukey_result)
    
    # Extract and display significant differences
    print("\n5. SIGNIFICANT DIFFERENCES IDENTIFIED:")
    print("-" * 80)
    tukey_df = pd.DataFrame(data=tukey_result.summary().data[1:], 
                           columns=tukey_result.summary().data[0])
    
    significant_pairs = tukey_df[tukey_df['reject'] == True]
    if len(significant_pairs) > 0:
        print(f"Found {len(significant_pairs)} significant pairwise differences:\n")
        for idx, row in significant_pairs.iterrows():
            print(f"  • {row['group1']} vs {row['group2']}: meandiff = {row['meandiff']}, p-value = {row['p-adj']}")
    else:
        print("No significant pairwise differences found.")
    
    return anova_stat, anova_p, tukey_result

In [4]:
# Load the dataset for One-Way ANOVA
df_oneway = pd.read_csv('Plant_Growth_New.csv')

print("DATASET FOR ONE-WAY ANOVA:")
print("=" * 80)
print(df_oneway)
print("\nDataset Shape:", df_oneway.shape)
print("\nDataset Info:")
print(df_oneway.info())
print("\nBasic Statistics:")
print(df_oneway.groupby('Treatment')['Height'].describe())

DATASET FOR ONE-WAY ANOVA:
                               Treatment  Height
0       low sun exposure and water daily       6
1       low sun exposure and water daily       6
2       low sun exposure and water daily       6
3       low sun exposure and water daily       5
4       low sun exposure and water daily       6
5   medium sun exposure and water daily        5
6   medium sun exposure and water daily        5
7   medium sun exposure and water daily        6
8   medium sun exposure and water daily        4
9   medium sun exposure and water daily        5
10     high sun exposure and water daily       6
11     high sun exposure and water daily       6
12     high sun exposure and water daily       7
13     high sun exposure and water daily       8
14     high sun exposure and water daily       7
15     low sun exposure and water weekly       3
16     low sun exposure and water weekly       4
17     low sun exposure and water weekly       4
18     low sun exposure and water weekly  

In [5]:
# Call the analyze_plant_growth function
anova_stat, anova_p, tukey_result = analyze_plant_growth(df_oneway)

ONE-WAY ANOVA ANALYSIS

1. GROUPED DATA BY TREATMENT:
--------------------------------------------------------------------------------
high sun exposure and water daily: [6, 6, 7, 8, 7]
high sun exposure and water weekly: [5, 6, 6, 7, 8]
low sun exposure and water daily: [6, 6, 6, 5, 6]
low sun exposure and water weekly: [3, 4, 4, 4, 5]
medium sun exposure and water daily : [5, 5, 6, 4, 5]
medium sun exposure and water weekly: [4, 4, 4, 4, 4]

2. PERFORMING ONE-WAY ANOVA:
--------------------------------------------------------------------------------
F-Statistic: 13.450000
P-value: 0.000003

3. INTERPRETATION OF ONE-WAY ANOVA RESULTS:
--------------------------------------------------------------------------------
✓ P-value (0.000003) < 0.05
✓ Reject the null hypothesis
✓ CONCLUSION: At least one treatment significantly affects plant growth.

4. TUKEY'S HSD (HONEST SIGNIFICANT DIFFERENCE) TEST:
--------------------------------------------------------------------------------
          

## Part 2: Two-Way ANOVA Analysis

**Objective:** Determine the effects of sunlight exposure and watering frequency on plant growth, and identify any interaction effects.

### Null and Alternate Hypotheses:

**Hypothesis Set 1 - Sunlight Exposure Effect:**
- **H₀₁ (Null):** There is no significant effect of Sunlight Exposure on plant growth.
- **H_a1 (Alternate):** There is significant effect of Sunlight Exposure on plant growth.

**Hypothesis Set 2 - Watering Frequency Effect:**
- **H₀₂ (Null):** There is no significant effect of Watering Frequency on plant growth.
- **H_a2 (Alternate):** There is significant effect of Watering Frequency on plant growth.

**Hypothesis Set 3 - Interaction Effect:**
- **H₀₃ (Null):** There is no significant interaction effect of Watering Frequency & Sunlight Exposure on plant growth.
- **H_a3 (Alternate):** There is significant interaction effect of Watering Frequency & Sunlight Exposure on plant growth.

In [6]:
# Load the dataset for Two-Way ANOVA
df_twoway = pd.read_csv('Plant_Growth_New_2.csv')

print("DATASET FOR TWO-WAY ANOVA:")
print("=" * 80)
print(df_twoway)
print("\nDataset Shape:", df_twoway.shape)
print("\nDataset Info:")
print(df_twoway.info())
print("\nBasic Statistics by Sun Exposure and Water Frequency:")
print(df_twoway.groupby(['Sun_Exposure', 'Water'])['Height'].describe())

DATASET FOR TWO-WAY ANOVA:
   Sun_Exposure   Water  Height
0           low   daily       6
1           low   daily       6
2           low   daily       6
3           low   daily       5
4           low   daily       6
5        medium   daily       5
6        medium   daily       5
7        medium   daily       6
8        medium   daily       4
9        medium   daily       5
10         high   daily       6
11         high   daily       6
12         high   daily       7
13         high   daily       8
14         high   daily       7
15          low  weekly       3
16          low  weekly       4
17          low  weekly       4
18          low  weekly       4
19          low  weekly       5
20       medium  weekly       4
21       medium  weekly       4
22       medium  weekly       4
23       medium  weekly       4
24       medium  weekly       4
25         high  weekly       5
26         high  weekly       6
27         high  weekly       6
28         high  weekly       7
29         hi

In [10]:
print("\n" + "=" * 80)
print("TWO-WAY ANOVA ANALYSIS")
print("=" * 80)

# Print the 6 statements for the hypotheses
print("\n1. HYPOTHESIS STATEMENTS:")
print("-" * 80)
print("Null Hypothesis H₀₁: There is no significant effect of Sunlight Exposure on plant growth.")
print("Alternate Hypothesis H_a1: There is significant effect of Sunlight Exposure on plant growth.")
print("\nNull Hypothesis H₀₂: There is no significant effect of Watering Frequency on plant growth.")
print("Alternate Hypothesis H_a2: There is significant effect of Watering Frequency on plant growth.")
print("\nNull Hypothesis H₀₃: There is no significant interaction effect of Watering Frequency & Sunlight Exposure on plant growth.")
print("Alternate Hypothesis H_a3: There is significant interaction effect of Watering Frequency & Sunlight Exposure on plant growth.")

# Perform Two-Way ANOVA Analysis
print("\n2. TWO-WAY ANOVA CALCULATIONS:")
print("-" * 80)

# Get unique levels
sun_exposure_levels = df_twoway['Sun_Exposure'].unique()
water_levels = df_twoway['Water'].unique()

# Calculate grand mean
grand_mean = df_twoway['Height'].mean()

# Calculate sum of squares
# Total Sum of Squares
sst = np.sum((df_twoway['Height'] - grand_mean) ** 2)

# Sum of Squares for Sun Exposure (factor A)
ss_sun = 0
for level in sun_exposure_levels:
    group_data = df_twoway[df_twoway['Sun_Exposure'] == level]['Height']
    group_mean = group_data.mean()
    group_size = len(group_data)
    ss_sun += group_size * (group_mean - grand_mean) ** 2

# Sum of Squares for Water (factor B)
ss_water = 0
for level in water_levels:
    group_data = df_twoway[df_twoway['Water'] == level]['Height']
    group_mean = group_data.mean()
    group_size = len(group_data)
    ss_water += group_size * (group_mean - grand_mean) ** 2

# Sum of Squares for interaction (A*B)
ss_ab = 0
for sun_level in sun_exposure_levels:
    for water_level in water_levels:
        group_data = df_twoway[(df_twoway['Sun_Exposure'] == sun_level) & 
                              (df_twoway['Water'] == water_level)]['Height']
        if len(group_data) > 0:
            group_mean = group_data.mean()
            sun_mean = df_twoway[df_twoway['Sun_Exposure'] == sun_level]['Height'].mean()
            water_mean = df_twoway[df_twoway['Water'] == water_level]['Height'].mean()
            group_size = len(group_data)
            ss_ab += group_size * (group_mean - sun_mean - water_mean + grand_mean) ** 2

# Sum of Squares Error (residual)
ss_error = sst - ss_sun - ss_water - ss_ab

# Degrees of freedom
n_total = len(df_twoway)
df_sun = len(sun_exposure_levels) - 1
df_water = len(water_levels) - 1
df_ab = df_sun * df_water
df_error = n_total - len(sun_exposure_levels) * len(water_levels)
df_total = n_total - 1

# Mean Squares
ms_sun = ss_sun / df_sun if df_sun > 0 else 0
ms_water = ss_water / df_water if df_water > 0 else 0
ms_ab = ss_ab / df_ab if df_ab > 0 else 0
ms_error = ss_error / df_error if df_error > 0 else 0

# F-statistics
from scipy.stats import f as f_dist
f_stat_1 = ms_sun / ms_error if ms_error > 0 else 0
f_stat_2 = ms_water / ms_error if ms_error > 0 else 0
f_stat_3 = ms_ab / ms_error if ms_error > 0 else 0

# P-values
p_value_1 = 1 - f_dist.cdf(f_stat_1, df_sun, df_error) if df_sun > 0 else 1
p_value_2 = 1 - f_dist.cdf(f_stat_2, df_water, df_error) if df_water > 0 else 1
p_value_3 = 1 - f_dist.cdf(f_stat_3, df_ab, df_error) if df_ab > 0 else 1

# Print ANOVA Table
print("\n3. TWO-WAY ANOVA TABLE:")
print("-" * 80)
print(f"{'Source':<20} {'SS':<12} {'DF':<8} {'MS':<12} {'F':<12} {'P-value':<12}")
print("-" * 76)
print(f"{'Sun Exposure':<20} {ss_sun:<12.4f} {df_sun:<8} {ms_sun:<12.4f} {f_stat_1:<12.4f} {p_value_1:<12.6f}")
print(f"{'Water':<20} {ss_water:<12.4f} {df_water:<8} {ms_water:<12.4f} {f_stat_2:<12.4f} {p_value_2:<12.6f}")
print(f"{'Interaction':<20} {ss_ab:<12.4f} {df_ab:<8} {ms_ab:<12.4f} {f_stat_3:<12.4f} {p_value_3:<12.6f}")
print(f"{'Error':<20} {ss_error:<12.4f} {df_error:<8} {ms_error:<12.4f}")
print(f"{'Total':<20} {sst:<12.4f} {df_total:<8}")

# Extract and display F-statistics and p-values
print("\n4. EXTRACTION OF F-STATISTICS AND P-VALUES:")
print("-" * 80)
print(f"\nSunlight Exposure (Sun_Exposure):")
print(f"  F-Statistic: {f_stat_1:.6f}")
print(f"  P-value: {p_value_1:.6f}")

print(f"\nWatering Frequency (Water):")
print(f"  F-Statistic: {f_stat_2:.6f}")
print(f"  P-value: {p_value_2:.6f}")

print(f"\nInteraction Effect (Sun_Exposure:Water):")
print(f"  F-Statistic: {f_stat_3:.6f}")
print(f"  P-value: {p_value_3:.6f}")


TWO-WAY ANOVA ANALYSIS

1. HYPOTHESIS STATEMENTS:
--------------------------------------------------------------------------------
Null Hypothesis H₀₁: There is no significant effect of Sunlight Exposure on plant growth.
Alternate Hypothesis H_a1: There is significant effect of Sunlight Exposure on plant growth.

Null Hypothesis H₀₂: There is no significant effect of Watering Frequency on plant growth.
Alternate Hypothesis H_a2: There is significant effect of Watering Frequency on plant growth.

Null Hypothesis H₀₃: There is no significant interaction effect of Watering Frequency & Sunlight Exposure on plant growth.
Alternate Hypothesis H_a3: There is significant interaction effect of Watering Frequency & Sunlight Exposure on plant growth.

2. TWO-WAY ANOVA CALCULATIONS:
--------------------------------------------------------------------------------

3. TWO-WAY ANOVA TABLE:
--------------------------------------------------------------------------------
Source               SS       

In [11]:
# Interpretation of results
print("\n5. INTERPRETATION OF TWO-WAY ANOVA RESULTS:")
print("-" * 80)

# H₀₁ Interpretation
print("\nH₀₁ - Sunlight Exposure Effect:")
if p_value_1 > 0.05:
    print(f"  ✗ P-value ({p_value_1:.6f}) > 0.05")
    print(f"  ✗ Fail to reject H₀₁")
    print(f"  ✗ CONCLUSION: Sunlight Exposure is NOT statistically significant.")
else:
    print(f"  ✓ P-value ({p_value_1:.6f}) ≤ 0.05")
    print(f"  ✓ Reject H₀₁")
    print(f"  ✓ CONCLUSION: Sunlight Exposure significantly affects plant height.")

# H₀₂ Interpretation
print("\nH₀₂ - Watering Frequency Effect:")
if p_value_2 > 0.05:
    print(f"  ✗ P-value ({p_value_2:.6f}) > 0.05")
    print(f"  ✗ Fail to reject H₀₂")
    print(f"  ✗ CONCLUSION: Watering Frequency is NOT statistically significant.")
else:
    print(f"  ✓ P-value ({p_value_2:.6f}) ≤ 0.05")
    print(f"  ✓ Reject H₀₂")
    print(f"  ✓ CONCLUSION: Watering Frequency significantly affects plant height.")

# H₀₃ Interpretation
print("\nH₀₃ - Interaction Effect (Sunlight × Watering Frequency):")
if p_value_3 > 0.05:
    print(f"  ✗ P-value ({p_value_3:.6f}) > 0.05")
    print(f"  ✗ Fail to reject H₀₃")
    print(f"  ✗ CONCLUSION: There is NO significant interaction effect.")
else:
    print(f"  ✓ P-value ({p_value_3:.6f}) ≤ 0.05")
    print(f"  ✓ Reject H₀₃")
    print(f"  ✓ CONCLUSION: There is a significant interaction effect between Sunlight Exposure and Watering Frequency.")


5. INTERPRETATION OF TWO-WAY ANOVA RESULTS:
--------------------------------------------------------------------------------

H₀₁ - Sunlight Exposure Effect:
  ✓ P-value (0.000002) ≤ 0.05
  ✓ Reject H₀₁
  ✓ CONCLUSION: Sunlight Exposure significantly affects plant height.

H₀₂ - Watering Frequency Effect:
  ✓ P-value (0.000527) ≤ 0.05
  ✓ Reject H₀₂
  ✓ CONCLUSION: Watering Frequency significantly affects plant height.

H₀₃ - Interaction Effect (Sunlight × Watering Frequency):
  ✗ P-value (0.120667) > 0.05
  ✗ Fail to reject H₀₃
  ✗ CONCLUSION: There is NO significant interaction effect.


In [ ]:
# Interpretation of results
print("\n5. INTERPRETATION OF TWO-WAY ANOVA RESULTS:")
print("-" * 80)

# H₀₁ Interpretation
print("\nH₀₁ - Sunlight Exposure Effect:")
if p_value_1 > 0.05:
    print(f"  ✗ P-value ({p_value_1:.6f}) > 0.05")
    print(f"  ✗ Fail to reject H₀₁")
    print(f"  ✗ CONCLUSION: Sunlight Exposure is NOT statistically significant.")
else:
    print(f"  ✓ P-value ({p_value_1:.6f}) ≤ 0.05")
    print(f"  ✓ Reject H₀₁")
    print(f"  ✓ CONCLUSION: Sunlight Exposure significantly affects plant height.")

# H₀₂ Interpretation
print("\nH₀₂ - Watering Frequency Effect:")
if p_value_2 > 0.05:
    print(f"  ✗ P-value ({p_value_2:.6f}) > 0.05")
    print(f"  ✗ Fail to reject H₀₂")
    print(f"  ✗ CONCLUSION: Watering Frequency is NOT statistically significant.")
else:
    print(f"  ✓ P-value ({p_value_2:.6f}) ≤ 0.05")
    print(f"  ✓ Reject H₀₂")
    print(f"  ✓ CONCLUSION: Watering Frequency significantly affects plant height.")

# H₀₃ Interpretation
print("\nH₀₃ - Interaction Effect (Sunlight × Watering Frequency):")
if p_value_3 > 0.05:
    print(f"  ✗ P-value ({p_value_3:.6f}) > 0.05")
    print(f"  ✗ Fail to reject H₀₃")
    print(f"  ✗ CONCLUSION: There is NO significant interaction effect.")
else:
    print(f"  ✓ P-value ({p_value_3:.6f}) ≤ 0.05")
    print(f"  ✓ Reject H₀₃")
    print(f"  ✓ CONCLUSION: There is a significant interaction effect between Sunlight Exposure and Watering Frequency.")

## Conclusion

This experiment successfully demonstrated the implementation and application of both **One-Way ANOVA** and **Two-Way ANOVA** statistical tests to analyze plant growth data under different treatment conditions.

### Key Findings:

**One-Way ANOVA Analysis:**
- Used to test whether different treatments (combinations of sunlight exposure and watering frequency) produce significantly different plant heights.
- Applied Tukey's HSD post-hoc test to identify which specific treatment groups differ significantly.
- This helps determine if treatments are effective on plant growth.

**Two-Way ANOVA Analysis:**
- Examined two independent variables (Sunlight Exposure and Watering Frequency) and their individual effects on plant growth.
- Tested for interaction effects between the two factors.
- Provided more detailed insights into which factors individually and collectively influence plant height.

### Statistical Concepts Applied:
1. **F-Statistic:** Measures the ratio of variance between groups to variance within groups
2. **P-value:** Determines statistical significance (α = 0.05)
3. **Tukey's HSD Test:** Post-hoc test for pairwise comparisons while controlling Type I error
4. **Interaction Effects:** Tests whether the effect of one factor depends on levels of another factor

### Applications:
- Plant biology and agricultural research
- Optimization of growing conditions
- Experimental design and hypothesis testing
- Decision-making based on statistical evidence